# Synchronization Primitives — Experiment Analysis

In [2]:
import subprocess
from pathlib import Path

import pandas as pd
import plotly.express as px

PYTHON_VERSIONS = ["3.12","3.13", "3.13t", "3.14", "3.14t"]

def run_benchmark(experiment: str, pythons: list = PYTHON_VERSIONS) -> pd.DataFrame:
    subprocess.run(
        ["nox", "-s", "experiments", "-p", *pythons, "--", experiment],
        check=True,
    )

    frames = []
    for python in pythons:
        path = Path(f"results/{experiment}_{python}.json")
        df = pd.read_json(path)
        df["python"] = python
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [9]:
df01 = run_benchmark("race_condition", pythons=["3.13"])
df01

nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/race_condition.py --output results/race_condition_3.13.json
nox > Session experiments-3.13 was successful in 35 seconds.


,python_version,case,tasks,result,expected,time_s,python
0,3.13.3,threading / unsafe,3,500,1500,0.628,3.13
1,3.13.3,threading / unsafe,2,500,1000,0.633,3.13
2,3.13.3,threading / unsafe,5,500,2500,0.635,3.13
3,3.13.3,threading / unsafe,4,500,2000,0.642,3.13
4,3.13.3,processes / unsafe,2,500,1000,0.954,3.13
5,3.13.3,processes / unsafe,3,500,1500,0.963,3.13
6,3.13.3,processes / unsafe,4,501,2000,0.976,3.13
7,3.13.3,processes / unsafe,5,501,2500,0.980,3.13
8,3.13.3,threading / safe,2,1000,1000,1.265,3.13
9,3.13.3,processes / safe,2,1000,1000,1.560,3.13


## 01. Race Condition

In [23]:
CASE_COLORS = {
    "threading / unsafe": "#93C4F9",  # light blue
    "threading / safe  ": "#1A6BC1",  # dark blue
    "processes / unsafe": "#F4B07A",  # light orange
    "processes / safe  ": "#C45A00",  # dark orange
}

px.bar(
    df01,
    x="tasks",
    y="time_s",
    color="case",
    color_discrete_map=CASE_COLORS,
    barmode="group",
    title="Execution time by concurrency level",
    labels={"time_s": "time (s)", "tasks": "concurrent tasks"},
)

## 02. Deadlock

In [1]:
import pandas as pd
import plotly.express as px
from experiments.deadlock import demo_deadlock, demo_fixed, demo_deadlock_3, demo_fixed_3

In [2]:
df_deadlock = demo_deadlock()
df_fixed = demo_fixed()

print("--- deadlock ---")
print(df_deadlock.to_string(index=False))
print("\n--- fixed (lock ordering) ---")
print(df_fixed.to_string(index=False))

--- deadlock ---
thread      event  time_s
    T1 acquired A   0.000
    T2 acquired B   0.001
    T1 DEADLOCKED   2.011
    T2 DEADLOCKED   2.011

--- fixed (lock ordering) ---
thread      event  time_s
    T1 acquired A   0.000
    T1 acquired B   0.505
    T1  completed   1.010
    T2 acquired A   1.010
    T2 acquired B   1.515
    T2  completed   2.020


In [4]:
EVENT_COLORS = {
    "acquired A":  "#1A6BC1",
    "acquired B":  "#93C4F9",
    "completed":   "#2CA02C",
    "DEADLOCKED":  "#D62728",
}

px.scatter(
    df_deadlock,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS,
    text="event",
    title="Lock acquisition timeline — deadlock scenario",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [5]:
px.scatter(
    df_fixed,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS,
    text="event",
    title="Lock acquisition timeline — fixed (lock ordering)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [3]:
df_deadlock_3 = demo_deadlock_3()
df_fixed_3 = demo_fixed_3()

print("--- deadlock (3 threads) ---")
print(df_deadlock_3.to_string(index=False))
print("\n--- fixed (3 threads) ---")
print(df_fixed_3.to_string(index=False))

--- deadlock (3 threads) ---
thread      event  time_s
    T1 acquired A   0.001
    T2 acquired B   0.001
    T3 acquired C   0.001
    T1 DEADLOCKED   3.016
    T2 DEADLOCKED   3.016
    T3 DEADLOCKED   3.016

--- fixed (3 threads) ---
thread      event  time_s
    T1 acquired A   0.000
    T2 acquired B   0.000
    T2 acquired C   0.501
    T2  completed   1.006
    T1 acquired B   1.006
    T1  completed   1.510
    T3 acquired A   1.510
    T3 acquired C   2.015
    T3  completed   2.521


In [4]:
EVENT_COLORS_3 = {
    "acquired A":  "#1A6BC1",
    "acquired B":  "#93C4F9",
    "acquired C":  "#BDD7EE",
    "completed":   "#2CA02C",
    "DEADLOCKED":  "#D62728",
}

px.scatter(
    df_deadlock_3,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS_3,
    text="event",
    title="Lock acquisition timeline — 3-thread deadlock (A→B, B→C, C→A)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

In [5]:
px.scatter(
    df_fixed_3,
    x="time_s",
    y="thread",
    color="event",
    color_discrete_map=EVENT_COLORS_3,
    text="event",
    title="Lock acquisition timeline — 3-thread fixed (global ordering A < B < C)",
    labels={"time_s": "time (s)", "thread": "thread"},
)

## 03. strace — uncontended vs contended

In [8]:
df03 = run_benchmark("strace_lock", pythons=["3.12", "3.13", "3.13t", "3.14", "3.14t"])
df03

nox > Running session experiments-3.12
nox > Creating virtual environment (uv) using python3.12 in .nox/experiments-3-12
nox > uv pip install pandas plotly
nox > python experiments/strace_lock.py --output results/strace_lock_3.12.json
nox > Session experiments-3.12 was successful in 11 seconds.
nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/strace_lock.py --output results/strace_lock_3.13.json
nox > Session experiments-3.13 was successful in 11 seconds.
nox > Running session experiments-3.13t
nox > Creating virtual environment (uv) using python3.13t in .nox/experiments-3-13t
nox > uv pip install pandas plotly
nox > python experiments/strace_lock.py --output results/strace_lock_3.13t.json
nox > Session experiments-3.13t was successful in 18 seconds.
nox > Running session experiments-3.14
nox > Creating virtual environment (uv) using python3.14 in .nox/exp

,threads,iterations,time_s,ns_per_acquire,python
0,1,500000,0.0490,98.0,3.12
1,2,500000,0.0820,164.0,3.12
2,3,500000,0.1253,250.6,3.12
3,4,500000,0.1468,293.5,3.12
4,5,500000,0.2005,401.1,3.12
5,6,500000,0.2468,493.6,3.12
6,7,500000,0.2537,507.4,3.12
7,8,500000,0.3001,600.3,3.12
8,1,500000,0.0444,88.8,3.13
9,2,500000,0.0867,173.4,3.13


In [9]:
PYTHON_COLORS = {
    "3.12":  "#2CA02C",  # green
    "3.13":  "#1A6BC1",  # dark blue
    "3.13t": "#93C4F9",  # light blue
    "3.14":  "#C45A00",  # dark orange
    "3.14t": "#F4B07A",  # light orange
}

thread_order = (
    df03.groupby("threads")["ns_per_acquire"]
    .mean()
    .sort_values()
    .index.tolist()
)

px.bar(
    df03,
    x="threads",
    y="ns_per_acquire",
    color="python",
    color_discrete_map=PYTHON_COLORS,
    barmode="group",
    category_orders={"threads": thread_order},
    title="Lock acquisition cost: uncontended (1 thread) vs contended",
    labels={"ns_per_acquire": "ns per acquire", "threads": "threads"},
    text="ns_per_acquire",
)

## 04. GIL Benchmark

In [23]:
df04 = run_benchmark("gil_benchmark", pythons=["3.12", "3.13", "3.13t", "3.14", "3.14t"])
df04

nox > Running session experiments-3.12
nox > Creating virtual environment (uv) using python3.12 in .nox/experiments-3-12
nox > uv pip install pandas plotly
nox > python experiments/gil_benchmark.py --output results/gil_benchmark_3.12.json
nox > Session experiments-3.12 was successful in 14 seconds.
nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/gil_benchmark.py --output results/gil_benchmark_3.13.json
nox > Session experiments-3.13 was successful in 13 seconds.
nox > Running session experiments-3.13t
nox > Creating virtual environment (uv) using python3.13t in .nox/experiments-3-13t
nox > uv pip install pandas plotly
nox > python experiments/gil_benchmark.py --output results/gil_benchmark_3.13t.json
nox > Session experiments-3.13t was successful in 18 seconds.
nox > Running session experiments-3.14
nox > Creating virtual environment (uv) using python3.14

,threads,time_s,python
0,1,0.412,3.12
1,2,0.378,3.12
2,3,0.361,3.12
3,4,0.363,3.12
4,5,0.366,3.12
5,6,0.366,3.12
6,7,0.371,3.12
7,8,0.370,3.12
8,1,0.404,3.13
9,2,0.366,3.13


In [24]:
PYTHON_COLORS = {
    "3.12":  "#2CA02C",  # green
    "3.13":  "#1A6BC1",  # dark blue
    "3.13t": "#93C4F9",  # light blue
    "3.14":  "#C45A00",  # dark orange
    "3.14t": "#F4B07A",  # light orange
}

px.line(
    df04.sort_values("threads"),
    x="threads",
    y="time_s",
    color="python",
    color_discrete_map=PYTHON_COLORS,
    markers=True,
    title="GIL benchmark: CPU-bound task — wall time vs thread count",
    labels={"time_s": "wall time (s)", "threads": "threads"},
)

## 05. Lock Levels

In [3]:
df05 = run_benchmark("lock_levels", pythons=["3.12", "3.13", "3.13t", "3.14", "3.14t"])
df05

nox > Running session experiments-3.12
nox > Creating virtual environment (uv) using python3.12 in .nox/experiments-3-12
nox > uv pip install pandas plotly
nox > python experiments/lock_levels.py --output results/lock_levels_3.12.json
nox > Session experiments-3.12 was successful in 13 seconds.
nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/lock_levels.py --output results/lock_levels_3.13.json
nox > Session experiments-3.13 was successful in 11 seconds.
nox > Running session experiments-3.13t
nox > Creating virtual environment (uv) using python3.13t in .nox/experiments-3-13t
nox > uv pip install pandas plotly
nox > python experiments/lock_levels.py --output results/lock_levels_3.13t.json
nox > Session experiments-3.13t was successful in 20 seconds.
nox > Running session experiments-3.14
nox > Creating virtual environment (uv) using python3.14 in .nox/exp

,lock,iterations,time_s,ns_per_acquire,python
0,threading.Lock,500000,0.0482,96.4,3.12
1,asyncio.Lock,500000,0.1628,325.7,3.12
2,multiprocessing.Lock,500000,0.5045,1009.0,3.12
3,threading.Lock,500000,0.0465,93.0,3.13
4,asyncio.Lock,500000,0.1542,308.4,3.13
5,multiprocessing.Lock,500000,0.4186,837.2,3.13
6,threading.Lock,500000,0.0336,67.2,3.13t
7,asyncio.Lock,500000,0.1452,290.4,3.13t
8,multiprocessing.Lock,500000,0.4160,832.0,3.13t
9,threading.Lock,500000,0.0268,53.7,3.14


In [ ]:
LOCK_COLORS = {
    "threading.Lock":       "#1A6BC1",
    "asyncio.Lock":         "#2CA02C",
    "multiprocessing.Lock": "#C45A00",
}

px.bar(
    df05.sort_values("lock"),
    x="python",
    y="ns_per_acquire",
    color="lock",
    color_discrete_map=LOCK_COLORS,
    barmode="group",
    title="Lock acquisition cost by level: threading vs asyncio vs multiprocessing",
    labels={"ns_per_acquire": "ns per acquire", "python_version": "Python version"},
    text="ns_per_acquire",
)

## 05b. Lock contention — scaling with worker count

In [15]:
df05b = run_benchmark("lock_contention", pythons=["3.13", "3.13t", "3.14", "3.14t"])
df05b

nox > Running session experiments-3.13
nox > Creating virtual environment (uv) using python3.13 in .nox/experiments-3-13
nox > uv pip install pandas plotly
nox > python experiments/lock_contention.py --output results/lock_contention_3.13.json
nox > Session experiments-3.13 was successful in 16 seconds.
nox > Running session experiments-3.13t
nox > Creating virtual environment (uv) using python3.13t in .nox/experiments-3-13t
nox > uv pip install pandas plotly
nox > python experiments/lock_contention.py --output results/lock_contention_3.13t.json
nox > Session experiments-3.13t was successful in 15 seconds.
nox > Running session experiments-3.14
nox > Creating virtual environment (uv) using python3.14 in .nox/experiments-3-14
nox > uv pip install pandas plotly
nox > python experiments/lock_contention.py --output results/lock_contention_3.14.json
nox > Session experiments-3.14 was successful in 15 seconds.
nox > Running session experiments-3.14t
nox > Creating virtual environment (uv) usi

,lock,workers,time_s,ns_per_acquire,python
0,threading.Lock,1,0.0188,94.0,3.13
1,threading.Lock,2,0.0318,158.8,3.13
2,threading.Lock,3,0.0558,279.1,3.13
3,threading.Lock,4,0.0436,218.0,3.13
4,threading.Lock,5,0.0515,257.6,3.13
...,...,...,...,...,...
91,multiprocessing.Lock,4,0.3699,73970.2,3.14t
92,multiprocessing.Lock,5,0.4186,83724.1,3.14t
93,multiprocessing.Lock,6,0.4469,89370.3,3.14t
94,multiprocessing.Lock,7,0.4748,94953.7,3.14t


In [16]:
PYTHON_COLORS = {
    "3.13":  "#1A6BC1",  # dark blue
    "3.13t": "#93C4F9",  # light blue
    "3.14":  "#C45A00",  # dark orange
    "3.14t": "#F4B07A",  # light orange
}

df05b_plot = df05b.copy()
df05b_plot["threading_model"] = df05b_plot["python"].map(
    lambda p: "no GIL" if p.endswith("t") else "GIL"
)

for lock_type in ["threading.Lock", "asyncio.Lock", "multiprocessing.Lock"]:
    px.line(
        df05b_plot[df05b_plot["lock"] == lock_type].sort_values("workers"),
        x="workers",
        y="ns_per_acquire",
        color="python",
        color_discrete_map=PYTHON_COLORS,
        facet_col="threading_model",
        markers=True,
        title=f"Lock contention: {lock_type}",
        labels={"ns_per_acquire": "ns per acquire", "workers": "workers"},
    ).show()